In [1]:
from aalibrary import config

## Setting An Environment

In [2]:
config.use_gcp_prod()

## Downloading A Raw File
Let's get a random raw file's parameters from the NCEI archive, and then use those parameters to download it using AALibrary.

In [3]:
from aalibrary.utils.ncei_utils import get_random_raw_file_from_ncei
from aalibrary.ingestion import download_raw_file_from_ncei

random_ship_name, random_survey_name, random_echosounder, random_raw_file = (
    get_random_raw_file_from_ncei(
    )
)

print(
    random_ship_name,
    random_survey_name,
    random_echosounder,
    random_raw_file,
    sep="\n",
)

download_raw_file_from_ncei(
    file_name=random_raw_file,
    file_type="raw",
    ship_name=random_ship_name,
    survey_name=random_survey_name,
    echosounder=random_echosounder,
    file_download_directory="./test_data_dir/",
    upload_to_gcp=False,
)

Bristol_Explorer
BE201201
ES60
L0063-D20120906-T195038-ES60.raw


## Upload A File/Folder to GCP

In [4]:
from aalibrary.egress import (
    upload_local_echosounder_files_from_directory_to_gcp_storage_bucket,
)

# This method uploads all files in the specified local directory to the correct
# location in the GCP storage bucket.
upload_local_echosounder_files_from_directory_to_gcp_storage_bucket(
    local_echosounder_directory_to_upload="./test_data_dir/",
    ship_name=random_ship_name,
    survey_name=random_survey_name,
    echosounder=random_echosounder,
    data_source="HDD",
)

Using GCP Project `ggn-nmfs-aa-prod-1` and GCP Storage Bucket `ggn-nmfs-aa-prod-1-data`
CHECKING DIRECTORY FOR RAW, IDX, BOT, AND NETCDF FILES...
FOUND 4 RAW FILES | 0 IDX FILES | 0 BOT FILES | 0 NETCDF FILES


Uploading raw files:   0%|          | 0/4 [00:00<?, ?it/s]

UPLOADING FILE `D20150818-T020051.raw` TO GCP AT `HDD/Bristol_Explorer/BE201201/ES60/data/raw/D20150818-T020051.raw`...


Uploading raw files:  25%|██▌       | 1/4 [00:15<00:47, 15.87s/it]

UPLOADED.
UPLOADING FILE `D20211126-T065737.raw` TO GCP AT `HDD/Bristol_Explorer/BE201201/ES60/data/raw/D20211126-T065737.raw`...


...

UPLOADED.
UPLOADING FILE `L0010-D20060603-T011017-ES60.raw` TO GCP AT `HDD/Bristol_Explorer/BE201201/ES60/data/raw/L0010-D20060603-T011017-ES60.raw`...


Uploading raw files:  75%|███████▌  | 3/4 [00:39<00:11, 11.91s/it]

UPLOADED.
UPLOADING FILE `L0063-D20120906-T195038-ES60.raw` TO GCP AT `HDD/Bristol_Explorer/BE201201/ES60/data/raw/L0063-D20120906-T195038-ES60.raw`...


Uploading raw files: 100%|██████████| 4/4 [00:46<00:00, 11.58s/it]

UPLOADED.
4 RAW FILES UPLOADED.
UPLOADS COMPLETE
RAW (4) | IDX (0) | BOT 0 | NETCDF (0)


In [5]:
# We can also upload a whole folder to GCP, and then use `gcp_utils` to move
# the files to the correct location in the GCP storage bucket.
from aalibrary import config
from aalibrary.egress import upload_folder_as_is_to_gcp
from aalibrary.utils.gcp_utils import rename_gcs_folder

upload_folder_as_is_to_gcp(
    local_folder_path="./test_data_dir/",
    destination_prefix="TEST/random_ship_name/",
)

rename_gcs_folder(
    gcp_bucket_name=config.GCP_PROD_BUCKET_NAME,
    old_folder_prefix="TEST/random_ship_name/",
    new_folder_prefix=f"TEST/{random_ship_name}/{random_survey_name}/{random_echosounder}/",
)

Using GCP Project `ggn-nmfs-aa-prod-1` and GCP Storage Bucket `ggn-nmfs-aa-prod-1-data`
Uploaded test_data_dir\L0063-D20120906-T195038-ES60.raw to TEST/random_ship_name/L0063-D20120906-T195038-ES60.raw
Uploaded test_data_dir\L0010-D20060603-T011017-ES60.raw to TEST/random_ship_name/L0010-D20060603-T011017-ES60.raw
Uploaded test_data_dir\D20150818-T020051.raw to TEST/random_ship_name/D20150818-T020051.raw
Uploaded test_data_dir\D20211126-T065737.raw to TEST/random_ship_name/D20211126-T065737.raw


Renaming GCS objects: 100%|██████████| 7/7 [00:03<00:00,  1.82it/s]

	Renamed TEST/random_ship_name/CCE09_Spring_Frosti-D20090502-T113648.bot to TEST/Bristol_Explorer/BE201201/ES60/CCE09_Spring_Frosti-D20090502-T113648.bot
	Renamed TEST/random_ship_name/CCE09_Spring_Frosti-D20090502-T113648.idx to TEST/Bristol_Explorer/BE201201/ES60/CCE09_Spring_Frosti-D20090502-T113648.idx
	Renamed TEST/random_ship_name/CCE09_Spring_Frosti-D20090502-T113648.raw to TEST/Bristol_Explorer/BE201201/ES60/CCE09_Spring_Frosti-D20090502-T113648.raw
	Renamed TEST/random_ship_name/D20150818-T020051.raw to TEST/Bristol_Explorer/BE201201/ES60/D20150818-T020051.raw
	Renamed TEST/random_ship_name/D20211126-T065737.raw to TEST/Bristol_Explorer/BE201201/ES60/D20211126-T065737.raw
	Renamed TEST/random_ship_name/L0010-D20060603-T011017-ES60.raw to TEST/Bristol_Explorer/BE201201/ES60/L0010-D20060603-T011017-ES60.raw
	Renamed TEST/random_ship_name/L0063-D20120906-T195038-ES60.raw to TEST/Bristol_Explorer/BE201201/ES60/L0063-D20120906-T195038-ES60.raw


## Creating A Tugboat Submission

In [6]:
# Normally, you would use the code below:
# from aalibrary.tugboat_api import TugboatAPI

# tb_api = TugboatAPI()
# tb_api.create_empty_submission_file(
#     # Where you want the file to be located.
#     file_download_directory=".",
#     # Rename it to your cruise/survey name, if possible.
#     file_name="tugboat_test_submission.json",
# )

In [7]:
# But for today, we will use a pre-filled Tugboat Submission File to
# demonstrate the Tugboat workflow. So we will skip the above step and use the
# code below to load in the pre-filled Tugboat Submission File.
from aalibrary.utils.tugboat_validations import TugboatValidator
tb_validator = TugboatValidator(
    submission_json_file_path="./test_tugboat_validation_submission.json"
)

Data successfully loaded for validation.
1 Error(s) found during validation:


 -  Field `seaArea` cannot be empty.


In [8]:
# Now we upload it to GCP as a backup.
from aalibrary.utils.cloud_utils import upload_file_to_gcp_bucket
from aalibrary.utils.helpers import parse_correct_gcp_storage_bucket_location

# Parse the correct location for the file.
gcp_storage_bucket_location = parse_correct_gcp_storage_bucket_location(
    file_name = "test_tugboat_validation_submission.json",
    file_type = "json",
    ship_name = random_ship_name,
    survey_name = random_survey_name,
    echosounder = random_echosounder,
    data_source = "TEST",
    is_survey_metadata = True, # NOTE: It is important to set this to True.
    debug = True,
)

# Upload the file to the correct location.
upload_file_to_gcp_bucket(
    blob_file_path = gcp_storage_bucket_location,
    local_file_path = "./test_tugboat_validation_submission.json",
    debug = True,
)

New data uploaded to TEST/Bristol_Explorer/BE201201/metadata/test_tugboat_validation_submission.json


In [9]:
# Normally, you would execute the code below, but we have a metadata UI
# on the way that will make this easier!


# from aalibrary.tugboat_api import TugboatAPI
# tb_api = TugboatAPI()
# tb_api.post_new_submission(
#     submission_json_file_path="./test_tugboat_validation_submission.json"
# )